In [1]:
import networkx as nx
import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sys
import os

In [2]:
root_dir = os.path.dirname(os.path.dirname(os.path.abspath("./test1_data.json")))
sys.path.append(root_dir)

from edge_sim_py import *

In [3]:
#Dispaly components
def Collect_Components()->dict:
    datasets={}
    #User
    datasets[f"{User.__name__}"]=[]
    for user in User.all():
        datasets[f"{user.__class__.__name__}"].append((user._to_dict()))
        
    datasets[f"{Application.__name__}"]=[]
    for app in Application.all():
        datasets[f"{app.__class__.__name__}"].append((app._to_dict()))
        
    datasets[f"{Service.__name__}"]=[]
    for service in Service.all():
        datasets[f"{service.__class__.__name__}"].append((service._to_dict()))
        
    datasets[f"{EdgeServer.__name__}"]=[]
    for server in EdgeServer.all():
        datasets[f"{server.__class__.__name__}"].append((server._to_dict()))
       
    datasets[f"{BaseStation.__name__}"]=[]    
    for station in BaseStation.all():
        datasets[f"{station.__class__.__name__}"].append((station._to_dict()))  
        
    datasets[f"{NetworkSwitch.__name__}"]=[]         
    for switch in NetworkSwitch.all():
        datasets[f"{switch.__class__.__name__}"].append((switch._to_dict()))
        
    datasets[f"{NetworkLink.__name__}"]=[]           
    for link in NetworkLink.all():
        datasets[f"{link.__class__.__name__}"].append((link._to_dict()))  

    return datasets
#

def printComponent(datasets,category:str):
    print(f"{category}:")
    for agent in datasets[category]:
        print(agent)
        print()
    
def printClass(category):
    print(category.__name__)
    #User
    for agent in category.all():
        print(agent._to_dict())
        print()
    

In [4]:
#1.导入测试
simulate = Simulator()
simulate.initialize(input_file="./test1_date.json")
datasets = Collect_Components()
printComponent(datasets,"User")
printComponent(datasets,"Application")
printComponent(datasets,"Service")
printComponent(datasets,"EdgeServer")
printComponent(datasets,"BaseStation")
printComponent(datasets,"NetworkSwitch")

User:
{'attributes': {'id': 1, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'1': 45}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'1': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 1}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 1}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 2, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'2': 30}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'2': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 2}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 2}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 3, 'coordinates': [0, 0], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'3': 30}, 'communication

In [5]:
printClass(Application)

Application
{'attributes': {'id': 1, 'label': 'app1', 'start_time': 1, 'delay': 0, 'self.status': 'init', 'delay_sla': 45}, 'relationships': {'services': [{'class': 'Service', 'id': 1}], 'users': [{'class': 'User', 'id': 1}]}}

{'attributes': {'id': 2, 'label': 'app2', 'start_time': 2, 'delay': 0, 'self.status': 'init', 'delay_sla': 30}, 'relationships': {'services': [{'class': 'Service', 'id': 2}], 'users': [{'class': 'User', 'id': 2}]}}

{'attributes': {'id': 3, 'label': 'app3', 'start_time': 3, 'delay': 0, 'self.status': 'init', 'delay_sla': 30}, 'relationships': {'services': [{'class': 'Service', 'id': 3}], 'users': [{'class': 'User', 'id': 3}]}}



In [6]:
#2.步进测试
# 先替换各类的步进函数
User.step = tools.User_step
User.set_communication_path = tools.User_path
Application.step = tools.Application_Step
Service.step = tools.Service_Step
Service.provision = tools.Service_Provision
NetworkFlow.step = tools.NetworkFlow_Step
EdgeServer.step = tools.EdgeServer_Step
EdgeServer.has_capacity_to_host = tools.has_capacity_to_host
Simulator.step = tools.Simulator_Step

In [7]:
# 然后进行仿真模拟
# 首先仿真器步数加1
simulate.schedule.steps+=1
simulate.schedule.time+=1
# 所有用户步进
for usr in User.all():
    usr.step()

printClass(User)

User
{'attributes': {'id': 1, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'1': 45}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'1': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 1}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 1}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 2, 'coordinates': [0, 3], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'2': 30}, 'communication_paths': {}, 'making_requests': {}, 'mobility_model_parameters': {}}, 'relationships': {'access_patterns': {'2': {'class': 'CircularDurationAndIntervalAccessPattern', 'id': 2}}, 'mobility_model': None, 'applications': [{'class': 'Application', 'id': 2}], 'base_station': {'class': 'BaseStation', 'id': 1}}}

{'attributes': {'id': 3, 'coordinates': [0, 0], 'coordinates_trace': [], 'delays': {}, 'delay_slas': {'3': 30}, 'communication_

In [8]:
print(simulate.current_services)

[Service_1]


In [ ]:

#假设将服务1部署在服务器1上（放置于等待队列中）
service1 = Service.all()[0]
server1 = EdgeServer.all()[0]
service1.provision(target_server = server1)  #compute shortest path while provision

In [10]:
#服务步进
for service in Service.all():
    service.step()
printClass(Service)
print(Service.all()[0]._Service__migrations)

Service
{'attributes': {'id': 1, 'label': 'service1', 'state': 0, '_available': False, 'flops_load': [100, 200], 'cpu_demand': 1, 'gpu_demand': 1, 'ssd_demand': 1, 'memory_demand': 2048, 'bw_demand': 100, 'delay': 0, 'finished': False, 'src': NetworkSwitch_1}, 'relationships': {'application': {'class': 'Application', 'id': 1}, 'server': {'class': 'EdgeServer', 'id': 1}}}

{'attributes': {'id': 2, 'label': 'service2', 'state': 0, '_available': False, 'flops_load': [200, 300], 'cpu_demand': 1, 'gpu_demand': 1, 'ssd_demand': 1, 'memory_demand': 2048, 'bw_demand': 100, 'delay': 0, 'finished': False, 'src': NetworkSwitch_1}, 'relationships': {'application': {'class': 'Application', 'id': 2}, 'server': None}}

{'attributes': {'id': 3, 'label': 'service3', 'state': 0, '_available': False, 'flops_load': [300, 400], 'cpu_demand': 1, 'gpu_demand': 1, 'ssd_demand': 1, 'memory_demand': 2048, 'bw_demand': 100, 'delay': 0, 'finished': False, 'src': NetworkSwitch_2}, 'relationships': {'application': 

In [11]:
# 应用步进
for app in Application.all():
    app.step()
printClass(Application)

Application
{'attributes': {'id': 1, 'label': 'app1', 'start_time': 1, 'delay': 0, 'self.status': 'wait', 'delay_sla': 45}, 'relationships': {'services': [{'class': 'Service', 'id': 1}], 'users': [{'class': 'User', 'id': 1}]}}

{'attributes': {'id': 2, 'label': 'app2', 'start_time': 2, 'delay': 0, 'self.status': 'init', 'delay_sla': 30}, 'relationships': {'services': [{'class': 'Service', 'id': 2}], 'users': [{'class': 'User', 'id': 2}]}}

{'attributes': {'id': 3, 'label': 'app3', 'start_time': 3, 'delay': 0, 'self.status': 'init', 'delay_sla': 30}, 'relationships': {'services': [{'class': 'Service', 'id': 3}], 'users': [{'class': 'User', 'id': 3}]}}



In [12]:
# 服务器步进
for server in EdgeServer.all():
    server.step()

printClass(EdgeServer)
#部署服务后服务没有和服务器相关联(在哪一步实现关联?)
'''
1.服务确定部署服务器后就关联:由provision实现
2.当服务器检测到等待队列中有待传输服务时实现关联：由EdgeServer.step实现
3.当服务对应流量下载完成后才实现关联:由NetworkFlow.step实现
采用第1种实现方式
'''

EdgeServer
{'attributes': {'id': 1, 'available': True, 'model_name': 'E5430', 'cpu': 8, 'gpu': 10, 'memory': 16384, 'disk': 131072, 'bw': 10000, 'cpu_demand': 1, 'gpu_demand': 1, 'memory_demand': 2048, 'disk_demand': 1, 'bw_demand': 100, 'resource_ratio': {'cpu': 0, 'gpu': 0, 'disk': 0, 'memory': 0, 'bw': 0}, 'Cabability': {'cpu': 100, 'gpu': 200, 'pcie': 1000}, 'coordinates': [5, 3], 'max_concurrent_layer_downloads': 10, 'active': True, 'power_model_parameters': {}}, 'relationships': {'power_model': None, 'base_station': None, 'network_switch': {'class': 'NetworkSwitch', 'id': 5}, 'services': [{'class': 'Service', 'id': 1}]}}

{'attributes': {'id': 2, 'available': True, 'model_name': 'E5430', 'cpu': 10, 'gpu': 10, 'memory': 16384, 'disk': 131072, 'bw': 10000, 'cpu_demand': 0, 'gpu_demand': 0, 'memory_demand': 0, 'disk_demand': 0, 'bw_demand': 0, 'resource_ratio': {'cpu': 0, 'gpu': 0, 'disk': 0, 'memory': 0, 'bw': 0}, 'Cabability': {'cpu': 200, 'gpu': 200, 'pcie': 2000}, 'coordinates':

'\n1.服务确定部署服务器后就关联:由provision实现\n2.当服务器检测到等待队列中有待传输服务时实现关联：由EdgeServer.step实现\n3.当服务对应流量下载完成后才实现关联:由NetworkFlow.step实现\n采用第1种实现方式\n'

In [13]:
printClass(Service)

Service
{'attributes': {'id': 1, 'label': 'service1', 'state': 0, '_available': False, 'flops_load': [100, 200], 'cpu_demand': 1, 'gpu_demand': 1, 'ssd_demand': 1, 'memory_demand': 2048, 'bw_demand': 100, 'delay': 0, 'finished': False, 'src': NetworkSwitch_1}, 'relationships': {'application': {'class': 'Application', 'id': 1}, 'server': {'class': 'EdgeServer', 'id': 1}}}

{'attributes': {'id': 2, 'label': 'service2', 'state': 0, '_available': False, 'flops_load': [200, 300], 'cpu_demand': 1, 'gpu_demand': 1, 'ssd_demand': 1, 'memory_demand': 2048, 'bw_demand': 100, 'delay': 0, 'finished': False, 'src': NetworkSwitch_1}, 'relationships': {'application': {'class': 'Application', 'id': 2}, 'server': None}}

{'attributes': {'id': 3, 'label': 'service3', 'state': 0, '_available': False, 'flops_load': [300, 400], 'cpu_demand': 1, 'gpu_demand': 1, 'ssd_demand': 1, 'memory_demand': 2048, 'bw_demand': 100, 'delay': 0, 'finished': False, 'src': NetworkSwitch_2}, 'relationships': {'application': 

In [14]:
print(EdgeServer.all()[0].download_queue)
print(EdgeServer.all()[0].download_queue[0]._to_dict())

[NetworkFlow_1]
{'id': 1, 'status': 'active', 'nodes': [{'type': 'NetworkSwitch', 'id': 1}, {'type': 'NetworkSwitch', 'id': 5}], 'path': [NetworkSwitch_1, NetworkSwitch_3, NetworkSwitch_4, NetworkSwitch_5], 'start': 2, 'end': None, 'data_to_transfer': 1, 'bandwidth': {1: None, 3: None, 4: None}, 'metadata': {'type': 'service', 'object': Service_1}}


In [15]:
#实现最短路径算法
"""
示例：
源：服务所在用户关联基站的交换机
目的：服务部署服务器相连交换机
                        path = nx.shortest_path(
                        G=topology,
                        source=origin.network_switch,
                        target=target.network_switch,  
                        weight="delay",
                        method="dijkstra",
                    )
目前仅基于链路时延实现最短路径判定
"""
print(service1.src)
print(server1.network_switch)
print(service1.path)



NetworkSwitch_1
NetworkSwitch_5
[NetworkSwitch_1, NetworkSwitch_3, NetworkSwitch_4, NetworkSwitch_5]


In [16]:
#网络流步进

In [17]:
#交换机步进

In [18]:

#链路步进